In [10]:
#%pip install pybind11
#%pip install riskfolio-lib


In [11]:
from multi_stock_downloader import MultiStockDownloader

import os
import warnings
from datetime import datetime
import riskfolio as rp

import numpy as np
import pandas as pd
import vectorbt as vbt
from vectorbt.portfolio.enums import Direction, SizeType
from vectorbt.portfolio.nb import order_nb, sort_call_seq_nb

vbt.settings.returns["year_freq"] = "252 days"

warnings.filterwarnings("ignore")


In [13]:
dl = MultiStockDownloader(
    tickers=['AAPL.O', 'AMZN.O', 'NVDA.O', 'MSFT.O', 'GOOGL.O', 'KO', 'NOC.N', 'PBR.N'],
    start_date='2023-01-01'
)
data = dl.download()
print(data.head())

Successfully downloaded 796 records
Successfully downloaded 796 records
Successfully downloaded 796 records
Successfully downloaded 796 records
Successfully downloaded 796 records
Successfully downloaded 796 records
Successfully downloaded 796 records
Successfully downloaded 796 records
            AAPL.O  AMZN.O  NVDA.O  MSFT.O  GOOGL.O     KO   NOC.N  PBR.N
Date                                                                     
2023-01-03  125.07   85.82  14.315  239.58    89.12  62.95  540.33    9.5
2023-01-04  126.36   85.14  14.749   229.1    88.08  62.92  526.45   9.71
2023-01-05  125.02   83.12  14.265  222.31     86.2   62.2  528.52  10.13
2023-01-06  129.62   86.08  14.859  224.93    87.34   63.4  521.42  10.28
2023-01-09  130.15   87.36  15.628  227.12    88.02  62.61  495.41  10.24


In [27]:
tickers = [
"JPM", "V", "MA", "BAC", "WFC", "GS", "MS", "AXP", "C"
]

data = yf.download(
    tickers, 
    start="2010-01-01", 
    end="2024-06-30", 
    auto_adjust=False
)["Close"].dropna()

[*********************100%***********************]  9 of 9 completed


In [28]:
num_tests = 2000
ann_factor = data.vbt.returns(freq="D").ann_factor

def pre_sim_func_nb(sc, every_nth):
    sc.segment_mask[:, :] = False
    sc.segment_mask[every_nth::every_nth, :] = True
    return ()

def pre_segment_func_nb(
    sc, find_weights_nb, history_len, ann_factor, num_tests, srb_sharpe
):
    if history_len == -1:
        close = sc.close[: sc.i, sc.from_col : sc.to_col]
    else:
        if sc.i - history_len <= 0:
            return (np.full(sc.group_len, np.nan),)
        close = sc.close[sc.i - history_len : sc.i, sc.from_col : sc.to_col]

    best_sharpe_ratio, weights = find_weights_nb(sc, close, num_tests)
    srb_sharpe[sc.i] = best_sharpe_ratio

    size_type = np.full(sc.group_len, SizeType.TargetPercent)
    direction = np.full(sc.group_len, Direction.LongOnly)
    temp_float_arr = np.empty(sc.group_len, dtype=np.float_)
    for k in range(sc.group_len):
        col = sc.from_col + k
        sc.last_val_price[col] = sc.close[sc.i, col]
    sort_call_seq_nb(sc, weights, size_type, direction, temp_float_arr)

    return (weights,)

def order_func_nb(oc, weights):
    col_i = oc.call_seq_now[oc.call_idx]
    return order_nb(
        weights[col_i],
        oc.close[oc.i, oc.col],
        size_type=SizeType.TargetPercent,
    )

In [29]:
def opt_weights(sc, close, num_tests):
    close = pd.DataFrame(close, columns=tickers)
    returns = close.pct_change().dropna()
    port = rp.Portfolio(returns=returns)
    port.assets_stats(method_mu="hist", method_cov="hist")
    w = port.optimization(model="Classic", rm="CDaR", obj="Sharpe", hist=True)
    weights = np.ravel(w.to_numpy())
    shp = rp.Sharpe(w, port.mu, cov=port.cov, returns=returns, rm="CDaR", alpha=0.05)
    return shp, weights

sharpe = np.full(data.shape, np.nan)


In [21]:
close = data
returns = close.pct_change().dropna()
port = rp.Portfolio(returns=returns)


In [30]:
pf = vbt.Portfolio.from_order_func(
    data,
    order_func_nb,
    pre_sim_func_nb=pre_sim_func_nb,
    pre_sim_args=(30,),
    pre_segment_func_nb=pre_segment_func_nb,
    pre_segment_args=(opt_weights, 252 * 4, ann_factor, num_tests, sharpe),
    cash_sharing=True,
    group_by=True,
    use_numba=False,
    freq="D"
)

pf.plot_cum_returns()

TypeError: Sharpe() got multiple values for argument 'returns'